# NYC 311 Preprocessing Starter (API-first, 2025 only)

This notebook downloads **only 2025** records from the NYC Open Data API, saves a **full raw archive with all available columns**, then builds a **smaller analysis table** for the rest of the pipeline.

Workflow:
1. Download 2025 month by month from the Socrata API
2. Save a **raw** Parquet cache with **all columns**
3. Build a **curated analysis** Parquet table
4. Filter to noise-related complaints
5. Export a frontend-friendly JSON


## 1. Imports and configuration

If you have a Socrata app token, set it in your environment as `SOCRATA_APP_TOKEN` before running the notebook.


In [1]:
from pathlib import Path
import calendar
import io
import json
import os
import time

import pandas as pd
import requests

# NYC Dataset API configuration
DATASET_ID = 'erm2-nwe9'
BASE_URL = f'https://data.cityofnewyork.us/resource/{DATASET_ID}.csv'
APP_TOKEN = os.getenv('SOCRATA_APP_TOKEN', '')

# Project paths
PROJECT_ROOT = Path('..').resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_DIR = PROJECT_ROOT / 'public' / 'data'

RAW_2025_PATH = RAW_DIR / 'nyc_311_2025_raw.parquet'
ANALYSIS_2025_PATH = RAW_DIR / 'nyc_311_2025_analysis.parquet'
NOISE_2025_PATH = RAW_DIR / 'nyc_311_noise_2025.parquet'
OUTPUT_PATH = OUTPUT_DIR / 'complaints.json'

# Download settings
YEAR = 2025
LIMIT = 50_000
SLEEP_SECONDS = 0.2
USE_CACHE = True

# Raw download strategy
# Use None to request all available columns from the API.
RAW_REQUEST_COLUMNS = None

# Curated analysis layer
ANALYSIS_COLUMNS = [
    'unique_key',
    'created_date',
    'closed_date',
    'agency',
    'complaint_type',
    'descriptor',
    'borough',
    'incident_zip',
    'city',
    'community_board',
    'open_data_channel_type',
    'latitude',
    'longitude',
    'status',
]

# Filter settings
NOISE_TYPES = [
    'Noise - Commercial',
    'Noise - Helicopter',
    'Noise - House of Worship',
    'Noise - Park',
    'Noise - Residential',
    'Noise - Street/Sidewalk',
    'Noise - Vehicle',
]

PROJECT_ROOT, RAW_2025_PATH, ANALYSIS_2025_PATH, NOISE_2025_PATH, OUTPUT_PATH

(WindowsPath('C:/Users/Kai/Desktop/Files/New York University/Spring 2026/CSGY-6313 Data Visualization/Final/nyc-311-noise-viz-starter'),
 WindowsPath('C:/Users/Kai/Desktop/Files/New York University/Spring 2026/CSGY-6313 Data Visualization/Final/nyc-311-noise-viz-starter/data/raw/nyc_311_2025_raw.parquet'),
 WindowsPath('C:/Users/Kai/Desktop/Files/New York University/Spring 2026/CSGY-6313 Data Visualization/Final/nyc-311-noise-viz-starter/data/raw/nyc_311_2025_analysis.parquet'),
 WindowsPath('C:/Users/Kai/Desktop/Files/New York University/Spring 2026/CSGY-6313 Data Visualization/Final/nyc-311-noise-viz-starter/data/raw/nyc_311_noise_2025.parquet'),
 WindowsPath('C:/Users/Kai/Desktop/Files/New York University/Spring 2026/CSGY-6313 Data Visualization/Final/nyc-311-noise-viz-starter/public/data/complaints.json'))

## 2. Helper functions

The main fix here is schema-safe reading and writing:

- every monthly CSV chunk is first read as **string**
- known datetime and numeric columns are converted explicitly
- remaining object columns are normalized to pandas **string** dtype before Parquet write

That avoids mixed-type Parquet failures like the `incident_zip` error you hit.


In [2]:
def month_window(year: int, month: int) -> tuple[str, str]:
    start = f'{year}-{month:02d}-01T00:00:00'
    if month == 12:
        end = f'{year + 1}-01-01T00:00:00'
    else:
        end = f'{year}-{month + 1:02d}-01T00:00:00'
    return start, end

def read_csv_chunk_as_strings(csv_text: str) -> pd.DataFrame:
    return pd.read_csv(
        io.StringIO(csv_text),
        dtype=str,
        low_memory=False,
        keep_default_na=True,
        na_values=['', 'null', 'NULL'],
    )

def fetch_month_311(
    year: int,
    month: int,
    columns: list[str] | None = None,
    limit: int = 50_000,
    sleep_seconds: float = 0.2,
    app_token: str = '',
) -> pd.DataFrame:
    start, end = month_window(year, month)
    where_clause = f"created_date >= '{start}' AND created_date < '{end}'"

    headers = {'X-App-Token': app_token} if app_token else {}
    frames = []
    offset = 0

    print(f'Fetching {calendar.month_name[month]} {year}...')

    while True:
        params = {
            '$where': where_clause,
            '$order': 'created_date',
            '$limit': limit,
            '$offset': offset,
        }

        if columns:
            params['$select'] = ','.join(columns)

        response = requests.get(BASE_URL, params=params, headers=headers, timeout=120)
        response.raise_for_status()

        chunk = read_csv_chunk_as_strings(response.text)
        if chunk.empty:
            break

        frames.append(chunk)
        print(f'  offset={offset:,} -> {len(chunk):,} rows, {chunk.shape[1]} columns')

        if len(chunk) < limit:
            break

        offset += limit
        time.sleep(sleep_seconds)

    if not frames:
        return pd.DataFrame(columns=columns or [])

    return pd.concat(frames, ignore_index=True)

def normalize_frame_for_parquet(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Datetime columns present in the 311 schema
    datetime_cols = [
        col for col in [
            'created_date',
            'closed_date',
            'due_date',
            'resolution_action_updated_date',
        ] if col in df.columns
    ]
    for col in datetime_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)

    # Numeric geospatial fields we actually need numerically
    numeric_cols = [
        col for col in [
            'latitude',
            'longitude',
            'x_coordinate_state_plane',
            'y_coordinate_state_plane',
        ] if col in df.columns
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Keep ZIP / IDs / categorical text as strings to avoid mixed object issues
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]):
            df[col] = df[col].astype('string')

    return df

def build_analysis_subset(df_raw: pd.DataFrame, analysis_columns: list[str]) -> tuple[pd.DataFrame, list[str]]:
    existing = [col for col in analysis_columns if col in df_raw.columns]
    missing = [col for col in analysis_columns if col not in df_raw.columns]
    subset = df_raw[existing].copy()
    subset = normalize_frame_for_parquet(subset)
    return subset, missing

## 3. Download 2025 from the API, or load the cached raw copy

This saves the **full 2025 archive** with all columns.  
Then it builds a smaller **analysis** table for the rest of the notebook.


In [3]:
RAW_DIR.mkdir(parents=True, exist_ok=True)

if USE_CACHE and RAW_2025_PATH.exists():
    df_raw = pd.read_parquet(RAW_2025_PATH)
    print(f'Loaded cached raw 2025 data from {RAW_2025_PATH}')
else:
    monthly_frames = []

    for month in range(1, 13):
        month_df = fetch_month_311(
            year=YEAR,
            month=month,
            columns=RAW_REQUEST_COLUMNS,
            limit=LIMIT,
            sleep_seconds=SLEEP_SECONDS,
            app_token=APP_TOKEN,
        )
        month_df['source_month'] = month
        monthly_frames.append(month_df)

    df_raw = pd.concat(monthly_frames, ignore_index=True)
    df_raw = normalize_frame_for_parquet(df_raw)

    df_raw.to_parquet(RAW_2025_PATH, index=False)
    print(f'Saved raw 2025 cache to {RAW_2025_PATH}')

df_raw = normalize_frame_for_parquet(df_raw)

if USE_CACHE and ANALYSIS_2025_PATH.exists():
    df_analysis = pd.read_parquet(ANALYSIS_2025_PATH)
    missing_analysis_columns = [col for col in ANALYSIS_COLUMNS if col not in df_analysis.columns]
    df_analysis = normalize_frame_for_parquet(df_analysis)
    print(f'Loaded cached analysis layer from {ANALYSIS_2025_PATH}')
else:
    df_analysis, missing_analysis_columns = build_analysis_subset(df_raw, ANALYSIS_COLUMNS)
    df_analysis.to_parquet(ANALYSIS_2025_PATH, index=False)
    print(f'Saved analysis layer to {ANALYSIS_2025_PATH}')

print(f'Loaded {len(df_raw):,} raw 2025 rows with {df_raw.shape[1]} columns')
print(f'Loaded {len(df_analysis):,} analysis rows with {df_analysis.shape[1]} columns')

if missing_analysis_columns:
    print('Missing requested analysis columns:', missing_analysis_columns)

Loaded cached raw 2025 data from C:\Users\Kai\Desktop\Files\New York University\Spring 2026\CSGY-6313 Data Visualization\Final\nyc-311-noise-viz-starter\data\raw\nyc_311_2025_raw.parquet
Loaded cached analysis layer from C:\Users\Kai\Desktop\Files\New York University\Spring 2026\CSGY-6313 Data Visualization\Final\nyc-311-noise-viz-starter\data\raw\nyc_311_2025_analysis.parquet
Loaded 3,654,959 raw 2025 rows with 45 columns
Loaded 3,654,959 analysis rows with 14 columns


## 4. Inspect the raw and analysis layers

In [4]:
print('Raw columns:')
df_raw.columns.tolist()

Raw columns:


['unique_key',
 'created_date',
 'closed_date',
 'agency',
 'agency_name',
 'complaint_type',
 'descriptor',
 'descriptor_2',
 'location_type',
 'incident_zip',
 'incident_address',
 'street_name',
 'cross_street_1',
 'cross_street_2',
 'intersection_street_1',
 'intersection_street_2',
 'address_type',
 'city',
 'landmark',
 'facility_type',
 'status',
 'due_date',
 'resolution_description',
 'resolution_action_updated_date',
 'community_board',
 'council_district',
 'police_precinct',
 'bbl',
 'borough',
 'x_coordinate_state_plane',
 'y_coordinate_state_plane',
 'open_data_channel_type',
 'park_facility_name',
 'park_borough',
 'vehicle_type',
 'taxi_company_borough',
 'taxi_pick_up_location',
 'bridge_highway_name',
 'bridge_highway_direction',
 'road_ramp',
 'bridge_highway_segment',
 'latitude',
 'longitude',
 'location',
 'source_month']

In [5]:
print('Raw created date range:')
print(df_raw['created_date'].min(), '->', df_raw['created_date'].max())
print()

print('Analysis columns:')
print(df_analysis.columns.tolist())
print()

print('Raw shape:', df_raw.shape)
print('Analysis shape:', df_analysis.shape)

Raw created date range:
2025-01-01 00:00:12+00:00 -> 2025-12-31 23:59:28+00:00

Analysis columns:
['unique_key', 'created_date', 'closed_date', 'agency', 'complaint_type', 'descriptor', 'borough', 'incident_zip', 'city', 'community_board', 'open_data_channel_type', 'latitude', 'longitude', 'status']

Raw shape: (3654959, 45)
Analysis shape: (3654959, 14)


## 5. Filter to noise-related complaints

The rest of the notebook uses the **analysis layer**, not the full raw table.


In [6]:
# df = df_analysis.copy()

# df = df[
#     df['complaint_type']
#     .astype(str)
#     .str.contains('|'.join(NOISE_KEYWORDS), case=False, na=False)
# ].copy()

# df = df.dropna(subset=['created_date', 'latitude', 'longitude', 'borough'])

# print(f'Rows after filtering to noise + required fields: {len(df):,}')
# df[['created_date', 'complaint_type', 'borough', 'latitude', 'longitude']].head()

In [7]:
df = df_analysis.copy()

complaint_type_normalized = (
    df['complaint_type']
    .astype('string')
    .str.strip()
)

noise_like_but_excluded = sorted(
    complaint_type_normalized[
        complaint_type_normalized.str.contains('Noise', case=False, na=False)
        & ~complaint_type_normalized.isin(NOISE_TYPES)
    ]
    .dropna()
    .unique()
    .tolist()
)

df = df[complaint_type_normalized.isin(NOISE_TYPES)].copy()

df = df.dropna(subset=['created_date', 'latitude', 'longitude', 'borough'])

print(f'Rows after filtering to explicit noise complaint types + required fields: {len(df):,}')
print('Kept complaint types:', sorted(df['complaint_type'].dropna().astype(str).unique().tolist()))

if noise_like_but_excluded:
    print('Noise-like complaint types excluded from export:', noise_like_but_excluded)

df[['created_date', 'complaint_type', 'borough', 'latitude', 'longitude']].head()

Rows after filtering to explicit noise complaint types + required fields: 774,818
Kept complaint types: ['Noise - Commercial', 'Noise - Helicopter', 'Noise - House of Worship', 'Noise - Park', 'Noise - Residential', 'Noise - Street/Sidewalk', 'Noise - Vehicle']
Noise-like complaint types excluded from export: ['Noise']


,created_date,complaint_type,borough,latitude,longitude
0,2025-01-01 00:00:12+00:00,Noise - Residential,BRONX,40.891872,-73.860168
1,2025-01-01 00:00:13+00:00,Noise - Residential,BROOKLYN,40.711210,-73.958973
2,2025-01-01 00:00:17+00:00,Noise - Residential,BRONX,40.891872,-73.860168
6,2025-01-01 00:00:42+00:00,Noise - Residential,BROOKLYN,40.666609,-73.920687
7,2025-01-01 00:00:50+00:00,Noise - Residential,BRONX,40.891872,-73.860168


## 6. Add time features

In [8]:
df['year'] = df['created_date'].dt.year
df['month'] = df['created_date'].dt.month
df['day_of_week'] = df['created_date'].dt.day_name()
df['hour'] = df['created_date'].dt.hour
df['date'] = df['created_date'].dt.date.astype(str)

df[['created_date', 'year', 'month', 'day_of_week', 'hour', 'date']].head()

,created_date,year,month,day_of_week,hour,date
0,2025-01-01 00:00:12+00:00,2025,1,Wednesday,0,2025-01-01
1,2025-01-01 00:00:13+00:00,2025,1,Wednesday,0,2025-01-01
2,2025-01-01 00:00:17+00:00,2025,1,Wednesday,0,2025-01-01
6,2025-01-01 00:00:42+00:00,2025,1,Wednesday,0,2025-01-01
7,2025-01-01 00:00:50+00:00,2025,1,Wednesday,0,2025-01-01


## 7. Placeholder neighborhood field

For now this still uses `incident_zip` as a placeholder. Later you can replace it with an NTA or community district spatial join.


In [9]:
def infer_neighborhood(row: pd.Series) -> str:
    value = row.get('incident_zip')
    if pd.isna(value) or value in [None, '']:
        return 'Unknown'
    return str(value)

df['neighborhood'] = df.apply(infer_neighborhood, axis=1)
df[['incident_zip', 'neighborhood']].head()

,incident_zip,neighborhood
0,10466,10466
1,11211,11211
2,10466,10466
6,11212,11212
7,10466,10466


## 8. Standardize frontend-ready fields

In [10]:
df_clean = pd.DataFrame({
    'id': df['unique_key'].astype(str),
    'createdAt': df['created_date'].dt.strftime('%Y-%m-%dT%H:%M:%S%z'),
    'complaintType': df['complaint_type'].astype(str),
    'borough': df['borough'].astype(str),
    'neighborhood': df['neighborhood'].astype(str),
    'latitude': df['latitude'].astype(float),
    'longitude': df['longitude'].astype(float),
    'status': df.get('status', pd.Series(['Unknown'] * len(df), index=df.index)).astype(str),
})

df_clean.head()

,id,createdAt,complaintType,borough,neighborhood,latitude,longitude,status
0,63577994,2025-01-01T00:00:12+0000,Noise - Residential,BRONX,10466,40.891872,-73.860168,Closed
1,63584096,2025-01-01T00:00:13+0000,Noise - Residential,BROOKLYN,11211,40.711210,-73.958973,Closed
2,63581480,2025-01-01T00:00:17+0000,Noise - Residential,BRONX,10466,40.891872,-73.860168,Closed
6,63572426,2025-01-01T00:00:42+0000,Noise - Residential,BROOKLYN,11212,40.666609,-73.920687,Closed
7,63581601,2025-01-01T00:00:50+0000,Noise - Residential,BRONX,10466,40.891872,-73.860168,Closed


## 9. Save outputs

- `nyc_311_2025_raw.parquet`: full raw 2025 archive with all available columns
- `nyc_311_2025_analysis.parquet`: curated analysis table
- `nyc_311_noise_2025.parquet`: richer filtered noise subset
- `complaints.json`: frontend-ready point records


In [11]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save the curated analysis layer again in case later cells enriched it
df_analysis.to_parquet(ANALYSIS_2025_PATH, index=False)
print(f'Saved curated analysis layer to {ANALYSIS_2025_PATH}')

df.to_parquet(NOISE_2025_PATH, index=False)
print(f'Saved cleaned noise subset to {NOISE_2025_PATH}')

records = df_clean.to_dict(orient='records')
OUTPUT_PATH.write_text(json.dumps(records, indent=2), encoding='utf-8')
print(f'Wrote {len(records):,} cleaned records to {OUTPUT_PATH}')

Saved curated analysis layer to C:\Users\Kai\Desktop\Files\New York University\Spring 2026\CSGY-6313 Data Visualization\Final\nyc-311-noise-viz-starter\data\raw\nyc_311_2025_analysis.parquet
Saved cleaned noise subset to C:\Users\Kai\Desktop\Files\New York University\Spring 2026\CSGY-6313 Data Visualization\Final\nyc-311-noise-viz-starter\data\raw\nyc_311_noise_2025.parquet
Wrote 774,818 cleaned records to C:\Users\Kai\Desktop\Files\New York University\Spring 2026\CSGY-6313 Data Visualization\Final\nyc-311-noise-viz-starter\public\data\complaints.json


## 10. Quick validation checks

In [12]:
print('Unique complaint types:', df_clean['complaintType'].nunique())
print('Borough counts:')
df_clean['borough'].value_counts(dropna=False).rename_axis('borough').to_frame('count')

Unique complaint types: 7
Borough counts:


,count
borough,
BRONX,272859
BROOKLYN,183220
MANHATTAN,160985
QUEENS,142260
STATEN ISLAND,15480
Unspecified,14


In [13]:
print('Missing values by output column:')
df_clean.isna().sum().to_frame('missing')

Missing values by output column:


,missing
id,0
createdAt,0
complaintType,0
borough,0
neighborhood,0
latitude,0
longitude,0
status,0


In [14]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PUBLIC_DATA_DIR = PROJECT_ROOT / "public" / "data"
PUBLIC_DATA_DIR.mkdir(parents=True, exist_ok=True)

NOISE_2025_PATH = RAW_DIR / "nyc_311_noise_2025.parquet"

df = pd.read_parquet(NOISE_2025_PATH).copy()

# Optional: keep only the columns you need for frontend exports
keep_cols = [
    "unique_key",
    "created_date",
    "complaint_type",
    "borough",
    "latitude",
    "longitude",
    "status",
    "incident_zip",
]
existing_cols = [c for c in keep_cols if c in df.columns]
df = df[existing_cols].copy()

df["created_date"] = pd.to_datetime(df["created_date"], utc=True, errors="coerce")
df = df.dropna(subset=["created_date", "latitude", "longitude", "borough", "complaint_type"])

# Placeholder neighborhood
df["neighborhood"] = df.get("incident_zip", pd.Series(index=df.index)).fillna("Unknown").astype(str)

# Time fields
df["year"] = df["created_date"].dt.year
df["month"] = df["created_date"].dt.month
df["hour"] = df["created_date"].dt.hour
df["day_of_week"] = df["created_date"].dt.day_name()

day_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]
df["day_of_week"] = pd.Categorical(df["day_of_week"], categories=day_order, ordered=True)

In [15]:
points = pd.DataFrame({
    "id": df["unique_key"].astype(str),
    "createdAt": df["created_date"].dt.strftime("%Y-%m-%dT%H:%M:%SZ"),
    "complaintType": df["complaint_type"].astype(str),
    "borough": df["borough"].astype(str),
    "neighborhood": df["neighborhood"].astype(str),
    "latitude": df["latitude"].astype(float),
    "longitude": df["longitude"].astype(float),
    "status": df.get("status", pd.Series(["Unknown"] * len(df), index=df.index)).astype(str),
    "year": df["year"].astype(int),
    "month": df["month"].astype(int),
    "hour": df["hour"].astype(int),
    "dayOfWeek": df["day_of_week"].astype(str),
})

(PUBLIC_DATA_DIR / "complaints_points.json").write_text(
    json.dumps(points.to_dict(orient="records")),
    encoding="utf-8",
)

224976711

In [16]:
temporal = (
    df.groupby(["day_of_week", "hour"], observed=True)
      .size()
      .reset_index(name="count")
)

temporal["dayOfWeek"] = temporal["day_of_week"].astype(str)
temporal["hour"] = temporal["hour"].astype(int)

(PUBLIC_DATA_DIR / "temporal_matrix.json").write_text(
    json.dumps(
        temporal[["dayOfWeek", "hour", "count"]].to_dict(orient="records")
    ),
    encoding="utf-8",
)

8869

In [17]:
complaint_types = (
    df.groupby(["complaint_type"], observed=True)
      .size()
      .reset_index(name="count")
      .rename(columns={"complaint_type": "complaintType"})
      .sort_values("count", ascending=False)
)

(PUBLIC_DATA_DIR / "complaint_type_counts.json").write_text(
    json.dumps(complaint_types.to_dict(orient="records")),
    encoding="utf-8",
)

403

In [18]:
neighborhoods = (
    df.groupby(["borough", "neighborhood"], observed=True)
      .size()
      .reset_index(name="count")
      .sort_values("count", ascending=False)
)

(PUBLIC_DATA_DIR / "neighborhood_stats.json").write_text(
    json.dumps(neighborhoods.to_dict(orient="records")),
    encoding="utf-8",
)

14678